### 1.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

tickers = ["MSFT", "AAPL", "GOOGL"]
prices = yf.download(tickers, start="2024-05-23", end="2026-05-23", auto_adjust=False)["Close"]
prices = prices[tickers]
print(prices.head())
print(prices.tail())
print("observations", len(prices))

[*********************100%***********************]  3 of 3 completed

Ticker            MSFT        AAPL       GOOGL
Date                                          
2024-05-23  427.000000  186.880005  173.550003
2024-05-24  430.160004  189.979996  174.990005
2024-05-28  430.320007  189.990005  176.399994
2024-05-29  429.170013  190.289993  175.899994
2024-05-30  414.670013  191.289993  172.110001
Ticker            MSFT        AAPL       GOOGL
Date                                          
2026-05-18  423.540009  297.839996  396.940002
2026-05-19  417.420013  298.970001  387.660004
2026-05-20  421.059998  302.250000  388.910004
2026-05-21  419.089996  304.989990  387.660004
2026-05-22  418.570007  308.820007  382.970001
observations 501


### 2.

In [2]:
caps = np.array([3.109, 4.536, 4.64])
weights = caps/caps.sum()
portfolio = 1000000.0
print("weights", weights.round(4))

weights [0.2531 0.3692 0.3777]


### 3.

In [3]:
returns = np.log(prices/prices.shift(1)).dropna().to_numpy()
alpha = 0.95

def full(x):
    return -portfolio*(weights*(np.exp(x) - 1)).sum(axis=1)

def linear(x):
    return -portfolio*(weights*x).sum(axis=1)

def quad(x):
    return -portfolio*(weights*(x + 0.5*x**2)).sum(axis=1)

def var(losses, level):
    ordered = np.sort(losses)
    return ordered[int(np.ceil(len(losses)*level)) - 1]

emp_full = var(full(returns), alpha)
emp_linear = var(linear(returns), alpha)
emp_quad = var(quad(returns), alpha)
print("daily returns", len(returns))
print("full", round(emp_full, 2))
print("linear", round(emp_linear, 2))
print("quadratic", round(emp_quad, 2))

daily returns 500
full 21043.05
linear 21415.73
quadratic 21038.61


### 4.

In [4]:
decay = 0.97
mean = np.zeros(3)
cov = np.zeros((3, 3))

for x in returns:
    # the cov update uses the mean from before the step so it goes first
    gap = x - mean
    cov = decay*cov + (1 - decay)*np.outer(gap, gap)
    mean = decay*mean + (1 - decay)*x

print("mean", mean.round(6))
print("daily volatility", np.sqrt(np.diag(cov)).round(6))
print("cov")
print(cov)

mean [0.000473 0.003636 0.003306]
daily volatility [0.017728 0.013881 0.021433]
cov
[[3.14265682e-04 3.43138895e-05 3.46330467e-06]
 [3.43138895e-05 1.92684217e-04 8.13758111e-05]
 [3.46330467e-06 8.13758111e-05 4.59380854e-04]]


### 5.

In [5]:
samples = 100000
rng = np.random.default_rng(42)
draws = rng.multivariate_normal(mean, cov, samples)

fit_full = var(full(draws), alpha)
fit_quad = var(quad(draws), alpha)

avg = -portfolio*weights.dot(mean)
sd = portfolio*np.sqrt(weights.dot(cov).dot(weights))
fit_linear = avg + sd*norm.ppf(alpha)

print("full", round(fit_full, 2))
print("linear", round(fit_linear, 2))
print("quadratic", round(fit_quad, 2))

full 16664.91
linear 16869.11
quadratic 16662.2


### 6.

In [6]:
print("operator, empirical, normal")
print("full", round(emp_full, 2), round(fit_full, 2))
print("linear", round(emp_linear, 2), round(fit_linear, 2))
print("quadratic", round(emp_quad, 2), round(fit_quad, 2))

operator, empirical, normal
full 21043.05 16664.91
linear 21415.73 16869.11
quadratic 21038.61 16662.2


The six estimates really fall into two groups rather than six separate answers, since within each
method the full and quadratic operators agree to within five dollars while the linear one sits a few
hundred above both. That ordering is forced by the algebra rather than by the data, because the
exponential of x minus one always exceeds x and so the linear operator overstates the loss on every
scenario, while the quadratic operator recovers that curvature and leaves out only a cubic term that
is negligible over a single day. For a book of plain stock with no option convexity in it there is
nothing else for a second order approximation to miss.

The gap between the two methods is much wider, and the obvious reading of it turns out to be only
half right. It would be easy to blame the whole 4,547 dollars on the realised returns having fatter
tails than a fitted normal, but the EWMA mean comes out at 0.271 percent a day, which on a million
dollars takes 2,711 dollars off the normal VaR before any question of shape arises and leaves about
1,836 dollars for the distribution to account for. The drift dominates because a decay of 0.97
carries a half life near 23 trading days, so the fitted mean describes the last month of the window
rather than the full two years and all three stocks were running hard into the end of it. What is
left over is the ordinary story, with the realised returns implying a portfolio volatility around
1.33 percent a day against the 1.19 percent that comes out of the EWMA covariance.